# ② 安装 iKataGo，整理引擎、配置和权重

先完成 `katago.ipynb`。本文件从上游下载 iKataGo，校验 SHA256，将第一步安装的引擎、权重和配置复制到对应目录。不会设置账号或启动服务。

上游 iKataGo 不附带在发布包中。需要网络；上游包改变时会停止安装。


In [ ]:
from pathlib import Path
import json
import subprocess

STATE = Path.home() / ".config/katago-rtx50/notebook.json"
if not STATE.is_file():
    raise RuntimeError("请先运行 katago.ipynb 完成第一步。")
paths = json.loads(STATE.read_text())
ENGINE_DIR = Path(paths["engineDir"])
WORK_DIR = Path(paths["workDir"])
# 如需换安装目录，可在这里修改 WORK_DIR；必须为空目录。
prepared = json.loads((ENGINE_DIR / "prepared.json").read_text())
MODEL = (ENGINE_DIR / prepared["model"]).resolve()
MODEL.relative_to(ENGINE_DIR.resolve())
print("iKataGo 安装目录：", WORK_DIR)


In [ ]:
if (WORK_DIR / "deployment.json").is_file() and (WORK_DIR / "service.sh").is_file():
    old = json.loads((WORK_DIR / "deployment.json").read_text())
    if old.get("engineSha256") != prepared["engineSha256"] or old.get("modelSha256") != prepared["modelSha256"]:
        raise RuntimeError("已有 iKataGo 使用不同引擎或权重，请选择新的 WORK_DIR。")
    print("保留已有 iKataGo 部署。")
else:
    subprocess.run(["bash", str(ENGINE_DIR / "deploy-ikatago.sh"),
                    "--work", str(WORK_DIR), "--model", str(MODEL), "--no-account"], check=True)
paths["workDir"] = str(WORK_DIR)
temporary = STATE.with_suffix(".tmp")
temporary.write_text(json.dumps(paths, indent=2) + "\n")
temporary.chmod(0o600)
temporary.replace(STATE)
print("安装完成。下一步：打开 ikatago start.ipynb")


## 文件位置

- 引擎：`work/data/bins/CUDA-Optimized-RTX50`
- 权重：`work/data/weights/`（保留原始文件名）
- 引擎配置：`work/data/configs/default_gtp.cfg`
- iKataGo 配置：`work/config/conf.yaml`
- 服务端：`work/ikatago-server`

部署目录可独立运行。此时尚未创建账号，也没有启动服务。
